# Домашняя работа: Беллмановские обновления в MountainCar и Acrobot

Этот ноутбук знакомит вас с двумя средами классического управления: `MountainCar-v0` и `Acrobot-v1`.
Задача — реализовать табличный Q-learning с беллмановскими обновлениями, сравнить стратегии обучения и сделать выводы по каждой среде.


## Учебные цели
- разработать дискретизаторы для разных непрерывных пространств состояний (2D у `MountainCar`, 6D у `Acrobot`)
- реализовать табличный Q-learning, параметризованный спецификацией среды
- сравнить влияние дискретизации и расписаний `epsilon` в двух независимых экспериментах
- сформулировать рекомендации по настройке беллмановских алгоритмов для новых сред


## Формат работы
- Выполняйте ноутбук сверху вниз; ячейки с `TODO` должны быть заполнены кодом или текстом.
- Если запускаете ноутбук в Colab, сначала выполните установку зависимостей (ячейка ниже).
- Фиксируйте все ключевые наблюдения: графики, таблицы, числовые метрики.
- В конце заполните секции с выводами и ответами на вопросы.



### Рекомендуемый порядок работы:
1. Сначала реализуйте класс `Discretizer` — это основа для всего остального
2. Протестируйте дискретизатор на простых примерах (создайте тестовые ячейки)
3. Реализуйте методы класса `QLearningAgent` по очереди
4. Запустите небольшое обучение (100-200 эпизодов) для отладки
5. Только после этого запускайте полные эксперименты

### Установка зависимостей
Запустите ячейку ниже только в окружениях без предустановленных библиотек.


In [ ]:
# Если работаете в Colab, раскомментируйте строки ниже.
# !pip install gymnasium numpy matplotlib tqdm -q


In [ ]:
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm


In [ ]:
SEED = 2025
random.seed(SEED)
np.random.seed(SEED)


## 1A. Разведка среды `MountainCar-v0`
Перед дискретизацией исследуйте пространство состояний и динамику наград. Соберите несколько эпизодов со случайной
политикой, зафиксируйте min/max по каждой координате и опишите, какие состояния посещаются чаще.


In [ ]:
# TODO: создайте среду MountainCar-v0, соберите случайные эпизоды и выведите статистику по состояниям
# Подсказки:
# 1. Создайте среду: gym.make('MountainCar-v0')
# 2. Соберите ~10 эпизодов со случайными действиями
# 3. Сохраните все наблюдения в список
# 4. Вычислите min/max для каждой координаты (позиция и скорость)
# 5. Посчитайте среднюю награду и длину эпизода

mc_env = gym.make('MountainCar-v0')
mc_rollouts, mc_rewards, mc_lengths = [], [], []

# TODO: напишите цикл по эпизодам
# for ep in range(10):
#     state, _ = mc_env.reset(seed=SEED + ep)
#     done = False
#     total_reward = 0.0
#     steps = 0
#     while not done:
#         mc_rollouts.append(state)
#         action = ...  # используйте mc_env.action_space.sample()
#         state, reward, terminated, truncated, _ = mc_env.step(action)
#         # обновите total_reward, steps и done

mc_env.close()

# TODO: преобразуйте mc_rollouts в numpy массив и выведите статистику
# mc_rollouts = np.asarray(mc_rollouts)
# position = mc_rollouts[:, 0]
# velocity = mc_rollouts[:, 1]
# print(f'Позиция: min={position.min():.3f}, max={position.max():.3f}')
# print(f'Скорость: min={velocity.min():.3f}, max={velocity.max():.3f}')

raise NotImplementedError("TODO: исследуйте наблюдения и награды MountainCar")

## 1B. Разведка среды `Acrobot-v1`
`Acrobot` имеет шесть признаков (cos/sin углов и угловые скорости). Исследуйте диапазоны и убедитесь,
что понимаете ограничения по скоростям. Зафиксируйте наблюдения, при которых эпизод завершается.


In [ ]:
# TODO: соберите случайные эпизоды в Acrobot-v1 и проанализируйте диапазоны наблюдений
# Подсказки:
# 1. Acrobot имеет 6 признаков: [cos(θ1), sin(θ1), cos(θ2), sin(θ2), θ1_dot, θ2_dot]
# 2. Соберите статистику по всем 6 измерениям
# 3. Обратите внимание на диапазоны угловых скоростей

acro_env = gym.make('Acrobot-v1')
acro_rollouts, acro_rewards, acro_lengths = [], [], []

# TODO: аналогично MountainCar, соберите эпизоды
# for ep in range(10):
#     state, _ = acro_env.reset(seed=SEED + 100 + ep)
#     ...

acro_env.close()

# TODO: выведите min/max для каждого из 6 признаков
# acro_rollouts = np.asarray(acro_rollouts)
# mins = acro_rollouts.min(axis=0)
# maxs = acro_rollouts.max(axis=0)
# for i, (mn, mx) in enumerate(zip(mins, maxs)):
#     print(f'feature {i}: [{mn:.3f}, {mx:.3f}]')

raise NotImplementedError("TODO: исследуйте динамику состояний Acrobot")

## 2. Спецификации сред
Чтобы переиспользовать код, опишем каждую среду через `EnvSpec`: диапазоны наблюдений, дефолтные бины и число шагов.
Вы можете добавлять свои спецификации (например, для модифицированных карт).


In [ ]:
@dataclass
class EnvSpec:
    name: str
    observation_ranges: Tuple[Tuple[float, float], ...]
    default_bins: Tuple[int, ...]
    max_steps: int
    reward_baseline: float
    description: str


ENV_SPECS: Dict[str, EnvSpec] = {
    "MountainCar-v0": EnvSpec(
        name="MountainCar-v0",
        observation_ranges=((-1.2, 0.6), (-0.07, 0.07)),
        default_bins=(24, 24),
        max_steps=200,
        reward_baseline=-110.0,
        description="2 измерения: позиция и скорость, цель — добраться до вершины",
    ),
    "Acrobot-v1": EnvSpec(
        name="Acrobot-v1",
        observation_ranges=((-1.0, 1.0), (-1.0, 1.0), (-1.0, 1.0), (-1.0, 1.0), (-4.0, 4.0), (-9.0, 9.0)),
        default_bins=(8, 8, 8, 8, 12, 12),
        max_steps=500,
        reward_baseline=-100.0,
        description="6 признаков: cos/sin углов и угловые скорости двух звеньев",
    ),
}

print("Доступные спецификации:")
for spec in ENV_SPECS.values():
    print(f"- {spec.name}: dims={len(spec.observation_ranges)}, default_bins={spec.default_bins}")


## 3. Универсальный дискретизатор
Реализуйте `Discretizer`, который принимает список диапазонов и соответствующее число бинов на измерение.
Класс должен уметь:
1. валидировать вход (одинаковая длина `ranges` и `bins`, бины ≥ 1);
2. клиппировать наблюдения к допустимым диапазонам;
3. переводить каждое измерение в индекс бина (`np.digitize` или ручные правила);
4. разворачивать индексы в один скаляр для обращения к Q-таблице любого размера.


In [ ]:
class Discretizer:
    """Преобразует непрерывное состояние в дискретный индекс согласно спецификации."""

    def __init__(self, ranges: Tuple[Tuple[float, float], ...], bins: Tuple[int, ...]):
        # TODO: проверьте корректность входных данных и предвычислите массивы границ
        # Подсказки:
        # 1. Проверьте, что len(ranges) == len(bins)
        # 2. Проверьте, что все значения bins >= 1
        # 3. Для каждого измерения создайте границы бинов через np.linspace
        #    Если bins[i] == 1, то границ не нужно (одно состояние)
        #    Если bins[i] > 1, создайте bins[i] - 1 границу между low и high
        
        self.ranges = tuple(ranges)
        self.bins = tuple(bins)
        self.edges: List[np.ndarray] = []
        
        # Пример для одного измерения:
        # for (low, high), b in zip(self.ranges, self.bins):
        #     if b == 1:
        #         self.edges.append(np.array([], dtype=np.float32))
        #     else:
        #         self.edges.append(np.linspace(low, high, b - 1, dtype=np.float32))
        
        raise NotImplementedError("TODO: подготовьте границы бинов для каждой размерности")

    def clip(self, state: np.ndarray) -> np.ndarray:
        """TODO: верните состояние, ограниченное указанными диапазонами."""
        # Подсказка: используйте np.clip для каждого измерения
        # state = np.asarray(state, dtype=np.float32)
        # clipped = []
        # for value, (low, high) in zip(state, self.ranges):
        #     clipped.append(float(np.clip(value, low, high)))
        # return np.asarray(clipped, dtype=np.float32)
        raise NotImplementedError("TODO: реализуйте клиппинг состояния для произвольного числа измерений")

    def to_bin_indices(self, state: np.ndarray) -> Tuple[int, ...]:
        """TODO: переведите наблюдение в кортеж индексов бинов."""
        # Подсказки:
        # 1. Сначала обрежьте состояние через self.clip()
        # 2. Для каждого измерения найдите индекс бина через np.digitize(value, edges)
        # 3. Убедитесь, что индекс находится в диапазоне [0, bins[i] - 1]
        # clipped = self.clip(state)
        # indices: List[int] = []
        # for value, edges, b in zip(clipped, self.edges, self.bins):
        #     idx = int(np.digitize(value, edges))
        #     idx = min(max(idx, 0), b - 1)
        #     indices.append(idx)
        # return tuple(indices)
        raise NotImplementedError("TODO: реализуйте маппинг состояния на индексы бинов")

    def flat_index(self, indices: Tuple[int, ...]) -> int:
        """TODO: превратите набор индексов в один скалярный индекс Q-таблицы."""
        # Подсказка: используйте np.ravel_multi_index(indices, self.bins)
        # Это преобразует многомерный индекс в одномерный
        # return int(np.ravel_multi_index(indices, self.bins))
        raise NotImplementedError("TODO: реализуйте развёрнутый индекс для произвольной размерности")

    @property
    def num_states(self) -> int:
        """TODO: верните общее число дискретных состояний."""
        # Подсказка: это произведение всех значений в self.bins
        # return int(np.prod(self.bins))
        raise NotImplementedError("TODO: посчитайте размер дискретного пространства состояний")

### Тестирование дискретизатора

После реализации `Discretizer` проверьте его работу на простом примере:

In [ ]:
# Тест дискретизатора (раскомментируйте после реализации класса)
# 
# # Простой тест: 2D пространство с 3x4 бинами
# test_disc = Discretizer(
#     ranges=((-1.0, 1.0), (-2.0, 2.0)),
#     bins=(3, 4)
# )
# 
# # Проверки:
# print(f"Всего состояний: {test_disc.num_states}")  # Должно быть 12
# 
# # Тест клиппинга
# state = np.array([0.5, -3.0])  # -3.0 выходит за границы
# clipped = test_disc.clip(state)
# print(f"Clipped state: {clipped}")  # Должно быть [0.5, -2.0]
# 
# # Тест индексации
# bin_indices = test_disc.to_bin_indices(np.array([0.0, 0.0]))
# print(f"Bin indices для [0.0, 0.0]: {bin_indices}")  # Примерно (1, 2)
# 
# flat = test_disc.flat_index(bin_indices)
# print(f"Flat index: {flat}")  # Число от 0 до 11
# 
# # Если всё работает корректно, вы увидите разумные значения
# print("✅ Discretizer работает корректно!")

## 4. Конфигурация и агент Q-learning
Теперь обобщим агента. `QLearningConfig` содержит имя среды, количество эпизодов, обучение и параметры исследования.
Агент должен уметь работать с любой спецификацией из `ENV_SPECS`, используя дефолтные бины или переопределённые в конфиге.


In [ ]:
@dataclass
class QLearningConfig:
    env_name: str = "MountainCar-v0"
    num_episodes: int = 4000
    max_steps: Optional[int] = None  # по умолчанию берём из EnvSpec
    learning_rate: float = 0.1
    discount: float = 0.99
    epsilon_start: float = 1.0
    epsilon_end: float = 0.05
    epsilon_decay_episodes: int = 2000
    bins: Optional[Tuple[int, ...]] = None
    seed: int = 42


class QLearningAgent:
    """Tabular Q-learning с поддержкой нескольких сред."""

    def __init__(self, env: gym.Env, config: QLearningConfig):
        if config.env_name not in ENV_SPECS:
            raise ValueError(f"Неизвестная среда: {config.env_name}")
        self.env = env
        self.config = config
        self.spec = ENV_SPECS[config.env_name]
        self.max_steps = config.max_steps or self.spec.max_steps
        bins = config.bins or self.spec.default_bins
        self.discretizer = Discretizer(self.spec.observation_ranges, bins)
        self.num_actions = env.action_space.n
        self.q_table = np.zeros((self.discretizer.num_states, self.num_actions), dtype=np.float32)
        self.rng = np.random.default_rng(config.seed)

    def epsilon_by_episode(self, episode: int) -> float:
        """TODO: реализуйте линейное расписание epsilon между start и end."""
        # Формула линейной интерполяции:
        # frac = min(episode / epsilon_decay_episodes, 1.0)
        # epsilon = epsilon_start + frac * (epsilon_end - epsilon_start)
        # 
        # Пример:
        # frac = min(episode / max(1, self.config.epsilon_decay_episodes), 1.0)
        # return self.config.epsilon_start + frac * (self.config.epsilon_end - self.config.epsilon_start)
        raise NotImplementedError("TODO: реализуйте расписание epsilon с обрезкой по epsilon_end")

    def state_index(self, obs: np.ndarray) -> int:
        """TODO: преобразуйте наблюдение в индекс состояния через дискретизатор."""
        # Подсказка: используйте self.discretizer.to_bin_indices() и self.discretizer.flat_index()
        # return self.discretizer.flat_index(self.discretizer.to_bin_indices(obs))
        raise NotImplementedError("TODO: используйте Discretizer для получения индекса состояния")

    def select_action(self, state_idx: int, epsilon: float) -> int:
        """TODO: реализуйте epsilon-greedy выбор действия."""
        # Подсказки:
        # 1. С вероятностью epsilon выберите случайное действие
        # 2. Иначе выберите действие с максимальным Q(state_idx, a)
        # 
        # if self.rng.random() < epsilon:
        #     return int(self.rng.integers(self.num_actions))
        # return int(np.argmax(self.q_table[state_idx]))
        raise NotImplementedError("TODO: реализуйте стратегию выбора действия")

    def greedy_action(self, state_idx: int) -> int:
        """Выбор лучшего действия без случайности (для оценки)."""
        return int(np.argmax(self.q_table[state_idx]))

    def bellman_update(self, s_idx: int, action: int, reward: float, next_idx: int, terminated: bool) -> None:
        """TODO: примените обновление Q(s,a) с обнулением бутстрэпа только при terminated."""
        # Формула Q-learning:
        # target = reward + γ * max_a' Q(s', a')  если не terminated
        # target = reward                          если terminated
        # Q(s, a) ← Q(s, a) + α * (target - Q(s, a))
        #
        # Пример:
        # target = reward if terminated else reward + self.config.discount * float(np.max(self.q_table[next_idx]))
        # td_error = target - float(self.q_table[s_idx, action])
        # self.q_table[s_idx, action] += self.config.learning_rate * td_error
        raise NotImplementedError("TODO: реализуйте беллмановское обновление для произвольного числа действий")

    def train(self) -> Dict[str, List[float]]:
        """TODO: запустите обучение и верните историю наград/длин/epsilon."""
        # Подсказки:
        # 1. Создайте словарь metrics = {'episode_reward': [], 'episode_length': [], 'epsilon': []}
        # 2. Для каждого эпизода:
        #    - Сбросьте среду: obs, _ = self.env.reset(seed=...)
        #    - Получите state_idx через self.state_index(obs)
        #    - Вычислите epsilon через self.epsilon_by_episode(episode)
        #    - Выполните до max_steps шагов:
        #      * Выберите действие через self.select_action(state_idx, epsilon)
        #      * Сделайте шаг: next_obs, reward, terminated, truncated, _ = self.env.step(action)
        #      * Обновите Q-таблицу через self.bellman_update(...)
        #      * Обновите state_idx
        #      * Если terminated или truncated, break
        #    - Сохраните метрики в словарь
        # 3. Верните metrics
        raise NotImplementedError("TODO: реализуйте цикл обучения Q-learning для self.max_steps")

    def evaluate(self, episodes: int = 5) -> Tuple[float, float]:
        """TODO: оцените жадную политику без случайности."""
        # Подсказки:
        # 1. Создайте новую среду (не self.env!)
        # 2. Прогоните episodes эпизодов только с greedy_action
        # 3. Верните средние (reward, length)
        #
        # env = gym.make(self.config.env_name)
        # rewards, lengths = [], []
        # try:
        #     for ep in range(episodes):
        #         obs, _ = env.reset(seed=self.config.seed + 10_000 + ep)
        #         state_idx = self.state_index(obs)
        #         total_reward = 0.0
        #         for t in range(self.max_steps):
        #             action = self.greedy_action(state_idx)
        #             obs, reward, terminated, truncated, _ = env.step(action)
        #             total_reward += float(reward)
        #             state_idx = self.state_index(obs)
        #             if terminated or truncated:
        #                 break
        #         rewards.append(total_reward)
        #         lengths.append(t + 1)
        # finally:
        #     env.close()
        # return float(np.mean(rewards)), float(np.mean(lengths))
        raise NotImplementedError("TODO: реализуйте оценку политики для выбранной среды")

### Проверка расписания epsilon

Перед запуском обучения полезно визуализировать, как будет меняться epsilon:

In [ ]:
# Визуализация расписания epsilon (раскомментируйте после реализации)
#
# # Создаём временный конфиг для проверки
# test_config = QLearningConfig(
#     epsilon_start=1.0,
#     epsilon_end=0.05,
#     epsilon_decay_episodes=2000,
#     num_episodes=4000
# )
# 
# # Создаём временную среду и агента
# temp_env = gym.make('MountainCar-v0')
# temp_agent = QLearningAgent(temp_env, test_config)
# 
# # Строим график
# episodes = np.arange(test_config.num_episodes)
# epsilons = [temp_agent.epsilon_by_episode(ep) for ep in episodes]
# 
# plt.figure(figsize=(10, 4))
# plt.plot(episodes, epsilons)
# plt.axhline(y=test_config.epsilon_end, color='r', linestyle='--', label='epsilon_end')
# plt.axvline(x=test_config.epsilon_decay_episodes, color='g', linestyle='--', label='decay_episodes')
# plt.xlabel('Эпизод')
# plt.ylabel('Epsilon')
# plt.title('Расписание epsilon')
# plt.legend()
# plt.grid(True)
# plt.show()
# 
# temp_env.close()
# print("✅ Расписание epsilon корректно!")

## 5. Оценка жадной политики (отдельная функция)
Иногда удобно отделить оценочную функцию от класса. Реализуйте вспомогательную функцию, которая прогоняет N
эпизодов без `epsilon` и возвращает средние награды/длины. Используйте её в экспериментах для валидации моделей.


In [ ]:
def evaluate_policy(env_name: str, agent: QLearningAgent, episodes: int = 5) -> Tuple[float, float]:
    """TODO: создайте новое окружение, прогоните жадную политику и верните средние метрики."""
    # Подсказка: это почти копия метода agent.evaluate(), но принимает env_name явно
    # env = gym.make(env_name)
    # rewards, lengths = [], []
    # try:
    #     for ep in range(episodes):
    #         obs, _ = env.reset(seed=agent.config.seed + 20_000 + ep)
    #         state_idx = agent.state_index(obs)
    #         total_reward = 0.0
    #         for t in range(agent.max_steps):
    #             action = agent.greedy_action(state_idx)
    #             obs, reward, terminated, truncated, _ = env.step(action)
    #             total_reward += float(reward)
    #             state_idx = agent.state_index(obs)
    #             if terminated or truncated:
    #                 break
    #         rewards.append(total_reward)
    #         lengths.append(t + 1)
    # finally:
    #     env.close()
    # return float(np.mean(rewards)), float(np.mean(lengths))
    raise NotImplementedError("TODO: реализуйте отдельную функцию оценки для любой среды")

In [ ]:
def moving_average(values: List[float], window: int = 100) -> np.ndarray:
    """Вычисляет скользящее среднее для сглаживания графиков."""
    arr = np.asarray(values, dtype=np.float32)
    if arr.size == 0:
        return arr
    if arr.size < window:
        return arr
    kernel = np.ones(window, dtype=np.float32) / window
    return np.convolve(arr, kernel, mode='valid')

## 6. Эксперимент A — `MountainCar`: влияние дискретизации
1. Используйте одну и ту же конфигурацию обучения, но разные сетки бинов (например, `(18, 18)` vs `(30, 30)`).
2. Зафиксируйте сиды для честного сравнения.
3. Логируйте скользящее среднее награды, длины эпизодов и итоговый greedy-score.
4. Сделайте выводы о том, сколько бинов нужно для устойчивого обучения в `MountainCar`.


In [ ]:
# TODO: запустите эксперимент A для MountainCar с >=2 конфигурациями дискретизации
# Подсказки:
# 1. Создайте словарь mc_results для хранения результатов
# 2. Определите конфигурации с разными bins, например:
#    - '18x18 (coarse)': bins=(18, 18)
#    - '30x30 (fine)': bins=(30, 30)
# 3. Для каждой конфигурации:
#    - Создайте среду и QLearningConfig
#    - Обучите агента
#    - Оцените результат
#    - Сохраните логи и метрики
raise NotImplementedError("TODO: сравните две сетки бинов для MountainCar")

In [ ]:
# TODO: визуализируйте награды/длины/epsilon для эксперимента A
# Подсказки:
# 1. Создайте фигуру с 2 графиками: награды и длины эпизодов
# 2. Используйте moving_average() для сглаживания кривых
# 3. Добавьте легенду с названиями конфигураций

raise NotImplementedError("TODO: постройте сравнение кривых для MountainCar")

### Выводы по эксперименту A

**Ответьте на следующие вопросы:**

1. **Скорость сходимости:** Какая сетка бинов научилась быстрее? Почему?
   - TODO: ваш ответ

2. **Финальная производительность:** Какая конфигурация достигла лучшей итоговой награды при оценке?
   - TODO: ваш ответ

3. **Стабильность:** Какая сетка показала меньше колебаний в награде на поздних этапах обучения?
   - TODO: ваш ответ

4. **Рекомендации:** Какое количество бинов вы бы порекомендовали для MountainCar и почему?
   - TODO: ваш ответ

## 7. Эксперимент B — `Acrobot`: расписания epsilon и стабильность
1. Зафиксируйте одну дискретизацию (например, дефолтную) и сравните как минимум два расписания `epsilon`
   (быстрое и медленное уменьшение).
2. Проанализируйте, как часто агент достигает целевой высоты и какие награды получает.
3. Сравните variance наград и сделайте выводы, какая стратегия исследования лучше для `Acrobot`.


In [ ]:
# TODO: запустите эксперимент B для Acrobot с разными расписаниями epsilon
# Подсказки:
# 1. Зафиксируйте дискретизацию (используйте default_bins из EnvSpec)
# 2. Сравните минимум 2 расписания epsilon:
#    - Быстрое: epsilon_decay_episodes = 600
#    - Медленное: epsilon_decay_episodes = 2500
# 3. Для Acrobot рекомендуется:
#    - learning_rate около 0.2
#    - discount около 0.995
#    - num_episodes >= 5000
# 4. Посчитайте success_rate: долю эпизодов с длиной < max_steps

raise NotImplementedError("TODO: сравните расписания epsilon для Acrobot")

In [ ]:
# TODO: визуализируйте результаты эксперимента B (награды, epsilon, длины)
# Подсказки:
# 1. Создайте 3 графика: награды, длины, epsilon
# 2. Используйте большее окно для moving_average (например, 200) из-за высокой дисперсии
# 3. На третьем графике покажите, как менялся epsilon во время обучения


raise NotImplementedError("TODO: визуализируйте сравнение стратегий исследования для Acrobot")

### Выводы по эксперименту B

**Ответьте на следующие вопросы:**

1. **Влияние расписания epsilon:** Как скорость уменьшения epsilon повлияла на обучение?
   - TODO: ваш ответ

2. **Success rate:** Какое расписание привело к большей доле успешных эпизодов? Почему?
   - TODO: ваш ответ

3. **Исследование vs эксплуатация:** Объясните баланс между exploration и exploitation в каждом расписании.
   - TODO: ваш ответ

4. **Рекомендации:** Какое расписание epsilon вы бы выбрали для Acrobot при ограниченном бюджете эпизодов (например, 3000)?
   - TODO: ваш ответ

## 8. Дополнительные исследования (опционально)

Если хотите углубиться в тему, попробуйте следующие эксперименты:

### Идея 1: Адаптивная дискретизация
- Используйте неравномерную сетку бинов (больше бинов в критичных областях пространства состояний)
- Для MountainCar: больше бинов около скорости = 0 и позиции = -0.5

### Идея 2: Добавление третьей среды
- Добавьте `CartPole-v1` или `Pendulum-v1` (с дискретизацией действий)
- Создайте новый EnvSpec и проверьте универсальность вашего кода

### Идея 3: Анализ Q-таблицы
- Визуализируйте Q-values для разных состояний
- Найдите "ключевые" состояния, где агент принимает критичные решения
- Для MountainCar постройте heatmap Q(position, velocity, action)

### Идея 4: Комбинированное расписание epsilon
- Попробуйте экспоненциальное затухание вместо линейного
- Или двухфазное расписание: быстро до 0.3, затем медленно до 0.05

### Идея 5: Влияние learning rate
- Сравните разные значения α (0.05, 0.1, 0.2, 0.5)
- Постройте график зависимости скорости сходимости от α

**Ваши эксперименты:**
TODO: опишите, что вы попробовали и к каким выводам пришли

## 9. Вопросы для самопроверки

1. **TODO:** Чем отличаются требования к дискретизации для MountainCar (2D) и Acrobot (6D)? Какие измерения требуют более тонкой дискретизации и почему?

2. **TODO:** Какое расписание `epsilon` показало себя лучшим в каждой среде и почему? Есть ли универсальная стратегия или нужно адаптировать под задачу?

3. **TODO:** Какие метрики вы использовали для контроля сходимости и стабильности? Почему недостаточно смотреть только на среднюю награду?

4. **TODO:** С какими проблемами вы столкнулись при применении табличного Q-learning? Какие среды не подойдут для этого подхода?